# Đo trước CometKiwi trên các cặp description đã gắn nhãn (Colab T4), bản 2

Notebook này chạy **trên Google Colab**. Nó chấm điểm chất lượng bản dịch (không cần bản tham chiếu) cho **4.291 cặp zh–vi đã có nhãn DUNG/SAI** (3.991 cặp tầng 1, 300 cặp tầng 2), để xem CometKiwi có tách được cặp đúng và cặp sai không. bge-m3 cosine đã thử và không đạt (AUC 0,57–0,63).

**Mô hình:** `Unbabel/wmt22-cometkiwi-da` (XLM-R large, khoảng 560 triệu tham số). Giấy phép CC BY-NC-SA 4.0: dùng cho nghiên cứu, không thương mại. Cần đã bấm **Acknowledge license** trên trang mô hình và có token Read.

**Vì sao có bản 2:** cài thư viện COMET thẳng vào Python của Colab làm vỡ các gói Colab cài sẵn (protobuf, transformers). Bản này cài COMET vào **một môi trường Python 3.11 riêng** trong `/content/kiwi_env`, không đụng tới các gói của Colab.

## Cách chạy
1. Nếu trước đó đã chạy bản 1 trên Colab: **Runtime → Disconnect and delete runtime** để có môi trường sạch.
2. **Runtime → Change runtime type → T4 GPU**. Mở notebook này (bản 2) và chạy lần lượt từng ô, **không cần Restart session**.
   - Ô cài môi trường mất khoảng 3–5 phút (tải PyTorch và COMET).
   - Ô đăng nhập: dán token Read khi được hỏi.
   - Ô tải dữ liệu: chọn file `data/eda/description/kiwi_do_truoc/kiwi_dau_vao.csv` trên máy.
   - Ô chấm điểm: lần đầu tải mô hình khoảng 2 GB, rồi chấm 2 × 4.291 cặp.
3. Ô cuối tự tải về `kiwi_ket_qua.csv`. Chép vào `data/eda/description/kiwi_do_truoc/` trên máy rồi báo Claude.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Môi trường riêng Python 3.11 cho COMET (uv tự tải Python 3.11), không đụng gói của Colab
!pip -q install uv
!uv venv /content/kiwi_env --python 3.11 -q
# setuptools < 81: torchmetrics 0.10.3 (bản COMET yêu cầu) cần pkg_resources, môi trường uv không có sẵn (03/10)
!uv pip install -q --python /content/kiwi_env/bin/python unbabel-comet pandas "setuptools<81"
!/content/kiwi_env/bin/python -c "import comet, torch, numpy; print('comet OK | torch', torch.__version__, '| numpy', numpy.__version__, '| GPU:', torch.cuda.is_available())"

In [ ]:
# Đăng nhập Hugging Face (token lưu vào ~/.cache/huggingface, môi trường riêng đọc lại được).
# Có Colab Secret tên HF_TOKEN thì dùng, không thì hỏi token.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Đã đăng nhập bằng Colab Secret HF_TOKEN")
except Exception:
    login()

In [ ]:
import pandas as pd
from google.colab import files

print("Chọn file kiwi_dau_vao.csv trên máy:")
up = files.upload()
open("/content/kiwi_dau_vao.csv", "wb").write(up[next(iter(up))])
df = pd.read_csv("/content/kiwi_dau_vao.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
print(f"{len(df):,} cặp")
print(df.groupby(["tap", "nhan"]).size())

In [ ]:
%%writefile /content/cham_kiwi.py
# Chạy trong môi trường riêng /content/kiwi_env: chấm CometKiwi 2 cách (chỉ giá trị; "tên: giá trị")
import time
import numpy as np
import pandas as pd
from comet import download_model, load_from_checkpoint

df = pd.read_csv("/content/kiwi_dau_vao.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
model = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))


def cham(src, mt, bs=64):
    t0 = time.time()
    out = model.predict([{"src": s, "mt": m} for s, m in zip(src, mt)], batch_size=bs, gpus=1, progress_bar=False)
    print(f"  {len(src):,} cặp, {time.time() - t0:.0f} giây", flush=True)
    return np.round(np.asarray(out.scores, dtype=float), 5)


df["kiwi_gia_tri"] = cham(df.gia_tri_zh.tolist(), df.gia_tri_vi.tolist())
df["kiwi_ten_gia_tri"] = cham(df.zh_ten.tolist(), df.vi_ten.tolist())
df.to_csv("/content/kiwi_ket_qua.csv", index=False, encoding="utf-8-sig")
print("Đã ghi /content/kiwi_ket_qua.csv")

In [ ]:
!/content/kiwi_env/bin/python /content/cham_kiwi.py

## Xem nhanh

AUC = xác suất một cặp SAI có điểm thấp hơn một cặp DUNG. 0,5 là đoán mò, 1 là tách hoàn hảo. bge-m3 được 0,57–0,63. Phân tích đầy đủ (bảng ngưỡng, ví dụ) Claude làm trên máy sau khi anh tải file về.

In [ ]:
import numpy as np


def auc(sai, dung):
    v = np.concatenate([sai, dung])
    r = pd.Series(v).rank().to_numpy()
    n1, n0 = len(sai), len(dung)
    return 1 - (r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


kq = pd.read_csv("/content/kiwi_ket_qua.csv", encoding="utf-8-sig", keep_default_na=False)
rows = []
for tap in ["tang1", "tang2"]:
    d = kq[kq.tap == tap]
    for col in ["kiwi_gia_tri", "kiwi_ten_gia_tri"]:
        s, g = d[d.nhan == "SAI"][col].to_numpy(float), d[d.nhan == "DUNG"][col].to_numpy(float)
        rows.append({"tap": tap, "cach": col, "tb_dung": round(g.mean(), 3), "tb_sai": round(s.mean(), 3),
                     "auc": round(auc(s, g), 3)})
display(pd.DataFrame(rows))

In [ ]:
files.download("/content/kiwi_ket_qua.csv")